<a href="https://colab.research.google.com/github/nailasalmayusroini/NLP-Project-Task1/blob/main/NLP_Project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Packages & Mount Drive

In [1]:
!pip install -q trafilatura scikit-learn tqdm

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/nlp_news_pipeline'
os.makedirs(f'{PROJECT_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/processed', exist_ok=True)
print('Project folder ready at:', PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder ready at: /content/drive/MyDrive/nlp_news_pipeline


# Authenticate with BigQuery

In [2]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

PROJECT_ID = 'nlp-news-pipeline'
client = bigquery.Client(project=PROJECT_ID)
print('Authenticated. Using project:', PROJECT_ID)

Authenticated. Using project: nlp-news-pipeline


Define the sample weeks (regular + targeted)

In [3]:
import datetime as dt

STUDY_START = dt.date(2021, 9, 1)
STUDY_END = dt.date(2026, 9, 1)

def week_bounds(any_date: dt.date):
    """Return (monday, sunday) for the week containing any_date."""
    monday = any_date - dt.timedelta(days=any_date.weekday())
    sunday = monday + dt.timedelta(days=6)
    return monday, sunday

#regular weeks: first Mon-Sun week of every month in range
regular_weeks = []
cursor = dt.date(STUDY_START.year, STUDY_START.month, 1)
while cursor <= STUDY_END:
    first_day = cursor
    monday, sunday = week_bounds(first_day)
    if monday < STUDY_START:
        monday = STUDY_START
    regular_weeks.append((monday, sunday, 'regular'))
    # advance to next month
    if cursor.month == 12:
        cursor = dt.date(cursor.year + 1, 1, 1)
    else:
        cursor = dt.date(cursor.year, cursor.month + 1, 1)

#targeted weeks: known major USD-relevant events
#format: (event_date, short_label)
TARGETED_EVENTS = [
    (dt.date(2022, 2, 24), 'Russia invades Ukraine'),
    (dt.date(2022, 3, 16), 'Fed first rate hike since 2018'),
    (dt.date(2023, 3, 10), 'Silicon Valley Bank collapse'),
    (dt.date(2023, 10, 7), 'Hamas attack on Israel / war outbreak'),
    (dt.date(2024, 11, 5), 'US presidential election'),
    (dt.date(2025, 1, 20), 'Trump inauguration'),
    (dt.date(2025, 2, 23), 'German federal election'),
    (dt.date(2025, 4, 2), "'Liberation Day' broad US tariffs announced"),
    (dt.date(2025, 6, 22), 'Iran-Israel conflict escalation / Strait of Hormuz tension'),
    (dt.date(2026, 1, 6), 'USD index hits multi-year low'),
    (dt.date(2026, 3, 4), 'Strait of Hormuz oil chokepoint tension'),
    (dt.date(2026, 5, 1), "Fed Chair Powell's term transition"),
]

targeted_weeks = []
for event_date, label in TARGETED_EVENTS:
    if STUDY_START <= event_date <= STUDY_END:
        monday, sunday = week_bounds(event_date)
        targeted_weeks.append((monday, sunday, 'targeted', label))

print(f'{len(regular_weeks)} regular weeks, {len(targeted_weeks)} targeted weeks defined.')

#combine, dedup overlapping weeks, group by year
all_weeks = {}
for monday, sunday, sample_type in regular_weeks:
    all_weeks[(monday, sunday)] = {'sample_type': sample_type, 'label': None}
for monday, sunday, sample_type, label in targeted_weeks:
    key = (monday, sunday)
    if key in all_weeks and all_weeks[key]['sample_type'] == 'regular':
        #targeted week happens to coincide with a regular week
        all_weeks[key] = {'sample_type': 'regular+targeted', 'label': label}
    else:
        all_weeks[key] = {'sample_type': 'targeted', 'label': label}

weeks_by_year = {}
for (monday, sunday), meta in sorted(all_weeks.items()):
    year = monday.year
    weeks_by_year.setdefault(year, []).append((monday, sunday, meta['sample_type'], meta['label']))

print(f'{len(all_weeks)} total unique weeks across {len(weeks_by_year)} years.')
for year, weeks in sorted(weeks_by_year.items()):
    print(f'  {year}: {len(weeks)} weeks')

61 regular weeks, 12 targeted weeks defined.
71 total unique weeks across 6 years.
  2021: 5 weeks
  2022: 14 weeks
  2023: 13 weeks
  2024: 14 weeks
  2025: 15 weeks
  2026: 10 weeks


# Pipeline function

In [4]:
import pandas as pd
import datetime as dt
import re
import os
import trafilatura
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from trafilatura.settings import use_config
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

CAMEO_ROOT_CODES = ['10', '13', '14', '15', '16', '17', '18', '20']
ACTOR_COUNTRIES = ['USA', 'CHN', 'RUS', 'GBR', 'DEU', 'FRA', 'JPN', 'EUR']
MAX_ARTICLES_PER_WEEK = 100
RANDOM_SEED = 42
MAX_WORKERS = 15
CHECKPOINT_EVERY = 500
FETCH_TIMEOUT_SECONDS = 10
SIMILARITY_THRESHOLD = 0.85

fetch_config = use_config()
fetch_config.set('DEFAULT', 'DOWNLOAD_TIMEOUT', str(FETCH_TIMEOUT_SECONDS))


def build_query_for_weeks(weeks):
    root_codes_str = ', '.join(f"'{c}'" for c in CAMEO_ROOT_CODES)
    countries_str = ', '.join(f"'{c}'" for c in ACTOR_COUNTRIES)
    date_conditions = ' OR '.join(
        f"SQLDATE BETWEEN {m.strftime('%Y%m%d')} AND {s.strftime('%Y%m%d')}"
        for m, s, _, _ in weeks
    )
    #partition-time filter
    #irrelevant days entirely instead of scanning the whole table every query.
    partition_conditions = ' OR '.join(
        f"DATE(_PARTITIONTIME) BETWEEN '{m.strftime('%Y-%m-%d')}' AND '{s.strftime('%Y-%m-%d')}'"
        for m, s, _, _ in weeks
    )
    return f'''
    SELECT
      GLOBALEVENTID, SQLDATE, Actor1CountryCode, Actor2CountryCode,
      EventCode, EventRootCode, GoldsteinScale, AvgTone, SOURCEURL
    FROM `gdelt-bq.gdeltv2.events_partitioned`
    WHERE ({partition_conditions})
      AND ({date_conditions})
      AND EventRootCode IN ({root_codes_str})
      AND (Actor1CountryCode IN ({countries_str}) OR Actor2CountryCode IN ({countries_str}))
      AND SOURCEURL IS NOT NULL
    '''


def tag_sample_type(date_str, weeks):
    d = dt.datetime.strptime(date_str, '%Y-%m-%d').date()
    for monday, sunday, sample_type, label in weeks:
        if monday <= d <= sunday:
            return sample_type, label, (monday, sunday)
    return 'unknown', None, None


def fetch_article_text(url):
    try:
        downloaded = trafilatura.fetch_url(url, config=fetch_config)
        if downloaded is None:
            return None
        return trafilatura.extract(downloaded)
    except Exception:
        return None


def clean_text(text):
    return re.sub(r'\s+', ' ', text).strip()


def process_year(year):
    print(f'===== Processing {year} =====')
    weeks_this_year = weeks_by_year.get(year, [])
    print(f'{len(weeks_this_year)} sample weeks for {year}.')
    if not weeks_this_year:
        print(f'No sample weeks for {year}, skipping.')
        return None

    #1. Query
    query = build_query_for_weeks(weeks_this_year)
    print(f'Running query for {year} ...')
    raw_year_df = client.query(query).to_dataframe()
    print(f'Retrieved {len(raw_year_df)} raw rows for {year} (before per-week cap).')

    raw_year_df['date'] = pd.to_datetime(raw_year_df['SQLDATE'], format='%Y%m%d').dt.strftime('%Y-%m-%d')

    tags = [tag_sample_type(d, weeks_this_year) for d in tqdm(raw_year_df['date'], desc=f'{year}: tagging')]
    raw_year_df['sample_type'] = [t[0] for t in tags]
    raw_year_df['event_label'] = [t[1] for t in tags]
    raw_year_df['_week_key'] = [t[2] for t in tags]

    print(f'Capping to at most {MAX_ARTICLES_PER_WEEK} articles per week...')
    capped_groups = []
    for week_key, group in tqdm(raw_year_df.groupby('_week_key'), desc=f'{year}: capping'):
        if len(group) > MAX_ARTICLES_PER_WEEK:
            group = group.sample(n=MAX_ARTICLES_PER_WEEK, random_state=RANDOM_SEED)
        capped_groups.append(group)
    year_df = pd.concat(capped_groups, ignore_index=True).drop(columns=['_week_key'])
    print(f'{year}: {len(year_df)} rows after capping (was {len(raw_year_df)}).')

    raw_out_path = f'{PROJECT_DIR}/data/raw/gdelt_events_{year}.csv'
    year_df.to_csv(raw_out_path, index=False)
    print('Saved to', raw_out_path)

    #2. Fetch article text
    articles_out_path = f'{PROJECT_DIR}/data/raw/articles_with_text_{year}.csv'
    if os.path.exists(articles_out_path):
        done_df = pd.read_csv(articles_out_path)
        done_urls = set(done_df['SOURCEURL'])
        print(f'Found existing checkpoint for {year} with {len(done_df)} rows — resuming.')
    else:
        done_df = pd.DataFrame()
        done_urls = set()

    todo_df = year_df[~year_df['SOURCEURL'].isin(done_urls)].reset_index(drop=True)
    print(f'{len(todo_df)} URLs left to fetch for {year}.')

    results = []

    def save_checkpoint():
        combined = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
        combined.to_csv(articles_out_path, index=False)

    if len(todo_df) > 0:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_row = {
                executor.submit(fetch_article_text, row['SOURCEURL']): row
                for _, row in todo_df.iterrows()
            }
            for i, future in enumerate(tqdm(as_completed(future_to_row), total=len(future_to_row), desc=f'{year}: fetching')):
                row = future_to_row[future]
                text = future.result()
                row_dict = row.to_dict()
                row_dict['article_text'] = text
                results.append(row_dict)
                if (i + 1) % CHECKPOINT_EVERY == 0:
                    save_checkpoint()
        save_checkpoint()

    full_df = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
    before = len(full_df)
    year_articles_df = full_df[full_df['article_text'].notna() & (full_df['article_text'].str.len() > 200)].copy()
    print(f'{year}: kept {len(year_articles_df)}/{before} rows with usable extracted text.')
    year_articles_df.to_csv(articles_out_path, index=False)

    #3. Clean & dedup
    year_articles_df['article_text'] = year_articles_df['article_text'].astype(str).map(clean_text)
    year_articles_df = year_articles_df.reset_index(drop=True)

    vectorizer = TfidfVectorizer(stop_words='english', max_features=20000)
    tfidf_matrix = vectorizer.fit_transform(year_articles_df['article_text'])
    sim_matrix = cosine_similarity(tfidf_matrix)

    n = len(year_articles_df)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        px, py = find(x), find(y)
        if px != py:
            parent[px] = py

    for i in range(n):
        for j in range(i + 1, n):
            if sim_matrix[i, j] >= SIMILARITY_THRESHOLD:
                union(i, j)

    clusters = {}
    for idx in range(n):
        root = find(idx)
        clusters.setdefault(root, []).append(idx)

    print(f'{year}: {len(clusters)} unique-article clusters from {n} raw articles.')

    def dedup_group(indices, df):
        subset = df.loc[indices]
        earliest_row = subset.sort_values('date').iloc[0]
        longest_row = subset.loc[subset['article_text'].str.len().idxmax()]
        merged = earliest_row.copy()
        merged['article_text'] = longest_row['article_text']
        merged['n_sources_merged'] = len(indices)
        return merged

    merged_rows = [dedup_group(idxs, year_articles_df) for idxs in clusters.values()]
    year_result_df = pd.DataFrame(merged_rows).reset_index(drop=True)

    year_final_path = f'{PROJECT_DIR}/data/processed/articles_deduped_{year}.csv'
    year_result_df.to_csv(year_final_path, index=False)
    print(f'{year}: saved {len(year_result_df)} deduplicated rows to {year_final_path}')
    print(year_result_df['sample_type'].value_counts())
    print(f'===== {year} done =====\n')
    return year_result_df

## Run 2021

In [5]:
result_2021 = process_year(2021)

===== Processing 2021 =====
5 sample weeks for 2021.
Running query for 2021 ...
Retrieved 195004 raw rows for 2021 (before per-week cap).


2021: tagging:   0%|          | 0/195004 [00:00<?, ?it/s]

Capping to at most 100 articles per week...


2021: capping:   0%|          | 0/5 [00:00<?, ?it/s]

2021: 500 rows after capping (was 195004).
Saved to /content/drive/MyDrive/nlp_news_pipeline/data/raw/gdelt_events_2021.csv
Found existing checkpoint for 2021 with 1909 rows — resuming.
204 URLs left to fetch for 2021.


2021: fetching:   0%|          | 0/204 [00:00<?, ?it/s]

ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.nytimes.com/2021/08/31/us/martinsville-seven-posthumous-pardons.html
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.bd-pratidin.com/editorial/2021/09/06/687997
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://sv.usembassy.gov/salvadoran-re-election-ruling-undermines-democracy/
ERROR:trafilatura.downloads:not a 200 response: 402 for URL https://www.telegraph.co.uk/films/0/parents-said-film-would-ruin-servant-shocked-1960s-britain/
ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://www.thegazette.com/health-care-medicine/spiking-covid-19-spread-isnt-cause-for-panic-iowa-gov-kim-reynolds-says/
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.calgarysun.com/news/national/top-clicks-the-week-that-was-in-viral-stor

2021: kept 1914/2113 rows with usable extracted text.
2021: 1654 unique-article clusters from 1914 raw articles.
2021: saved 1654 deduplicated rows to /content/drive/MyDrive/nlp_news_pipeline/data/processed/articles_deduped_2021.csv
sample_type
regular    1654
Name: count, dtype: int64
===== 2021 done =====



## Run 2022

In [6]:
result_2022 = process_year(2022)

===== Processing 2022 =====
14 sample weeks for 2022.
Running query for 2022 ...
Retrieved 804727 raw rows for 2022 (before per-week cap).


2022: tagging:   0%|          | 0/804727 [00:00<?, ?it/s]

Capping to at most 100 articles per week...


2022: capping:   0%|          | 0/14 [00:00<?, ?it/s]

2022: 1400 rows after capping (was 804727).
Saved to /content/drive/MyDrive/nlp_news_pipeline/data/raw/gdelt_events_2022.csv
Found existing checkpoint for 2022 with 500 rows — resuming.
900 URLs left to fetch for 2022.


2022: fetching:   0%|          | 0/900 [00:00<?, ?it/s]

ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://www.derwesten.de/region/nrw-mord-tot-tod-claudia-otto-detlef-m-festnahme-lohmar-strafe-id235180117.html
ERROR:trafilatura.downloads:not a 200 response: 400 for URL https://menafn.com/1104099376/International-partners-express-willingness-to-continue-to-assist-Ukraine-NBU&source=138
ERROR:trafilatura.downloads:download error: http://special.tass.ru/mezhdunarodnaya-panorama/14476031 HTTPConnectionPool(host='special.tass.ru', port=80): Max retries exceeded with url: /mezhdunarodnaya-panorama/14476031 (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x7ed66b0078a0>: Failed to resolve 'special.tass.ru' ([Errno -2] Name or service not known)"))
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.newscentermaine.com/article/news/crime/former-brunswick-resi

2022: kept 856/1400 rows with usable extracted text.
2022: 832 unique-article clusters from 856 raw articles.
2022: saved 832 deduplicated rows to /content/drive/MyDrive/nlp_news_pipeline/data/processed/articles_deduped_2022.csv
sample_type
regular     715
targeted    117
Name: count, dtype: int64
===== 2022 done =====



## Run 2023

In [7]:
result_2023 = process_year(2023)

===== Processing 2023 =====
13 sample weeks for 2023.
Running query for 2023 ...
Retrieved 710412 raw rows for 2023 (before per-week cap).


2023: tagging:   0%|          | 0/710412 [00:00<?, ?it/s]

Capping to at most 100 articles per week...


2023: capping:   0%|          | 0/13 [00:00<?, ?it/s]

2023: 1300 rows after capping (was 710412).
Saved to /content/drive/MyDrive/nlp_news_pipeline/data/raw/gdelt_events_2023.csv
1300 URLs left to fetch for 2023.


ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.nytimes.com/2023/02/01/world/europe/ukraine-russia-offensive.html
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.bizpacreview.com/2023/01/31/michigan-man-convicted-of-joining-isis-training-in-terrorist-tactics-1329100/
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.business-standard.com/article/companies/patent-infringement-hc-orders-triveni-chemicals-to-pay-rs-2-cr-to-pfizer-123020201682_1.html
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.khou.com/article/news/crime/norman-wilkerson-guilty-sexual-abuse-children/285-bcf93345-3b4d-4920-bd11-55073bc00fbc


2023: fetching:   0%|          | 0/1300 [00:00<?, ?it/s]

ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.downloads:not a 200 response: 401 for URL https://www.rbc.ru/politics/01/02/2023/63da209e9a7947d6f6a749f6
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://newsghana.com.gh/african-americans-and-the-united-states-civil-war/
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.dnevnik.bg/sviat/voinata_v_ukraina/2023/02/02/4445567_evroparlamentut_prizova_za_sankcionirane_na_lukoil_i/
ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://www.irishamerica.com/2023/01/troubles-bill-protest-as-bloody-sunday-victims-remembered-in-westminster/
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.downloads:download error: http://special.tass.ru/mezhdunarodnaya-panorama/16946201 HTTPConnectionPool(host='special.tas

2023: kept 794/1300 rows with usable extracted text.
2023: 768 unique-article clusters from 794 raw articles.
2023: saved 768 deduplicated rows to /content/drive/MyDrive/nlp_news_pipeline/data/processed/articles_deduped_2023.csv
sample_type
regular     650
targeted    118
Name: count, dtype: int64
===== 2023 done =====



## Run 2024

In [8]:
result_2024 = process_year(2024)

===== Processing 2024 =====
14 sample weeks for 2024.
Running query for 2024 ...
Retrieved 691903 raw rows for 2024 (before per-week cap).


2024: tagging:   0%|          | 0/691903 [00:00<?, ?it/s]

Capping to at most 100 articles per week...


2024: capping:   0%|          | 0/14 [00:00<?, ?it/s]

2024: 1400 rows after capping (was 691903).
Saved to /content/drive/MyDrive/nlp_news_pipeline/data/raw/gdelt_events_2024.csv
1400 URLs left to fetch for 2024.


ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://www.aol.com/news/taiwan-china-surely-reunified-says-052709739.html


2024: fetching:   0%|          | 0/1400 [00:00<?, ?it/s]

ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.al.com/news/2024/01/alabama-man-brother-sentenced-to-life-in-federal-prison-for-murder-for-hire-for-ex-wifes-2017-shooting-death.html
ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://foxwilmington.com/entertainment/friends-star-david-schwimmer-calls-out-skeptics-of-hamas-sexual-assaults-where-is-their-outrage/
ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://www.journal-spectator.com/news/article_1a2b187c-a9c3-11ee-bc6c-7b4979b8c697.html
ERROR:trafilatura.downloads:not a 200 response: 402 for URL https://www.telegraph.co.uk/news/2024/01/04/snp-access-middle-class-students-less-likely-law-school/
ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://republika.mk/vesti/svet/uapsen-petiot-osomnichen-za-planirane-napad-na-katedralata-vo-keln/
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.dailymail.co.uk/news/article-12915165/Sam-Brinton-Biden

2024: kept 935/1400 rows with usable extracted text.
2024: 926 unique-article clusters from 935 raw articles.
2024: saved 926 deduplicated rows to /content/drive/MyDrive/nlp_news_pipeline/data/processed/articles_deduped_2024.csv
sample_type
regular     860
targeted     66
Name: count, dtype: int64
===== 2024 done =====



## Run 2025

In [9]:
result_2025 = process_year(2025)

===== Processing 2025 =====
15 sample weeks for 2025.
Running query for 2025 ...
Retrieved 699290 raw rows for 2025 (before per-week cap).


2025: tagging:   0%|          | 0/699290 [00:00<?, ?it/s]

Capping to at most 100 articles per week...


2025: capping:   0%|          | 0/15 [00:00<?, ?it/s]

2025: 1500 rows after capping (was 699290).
Saved to /content/drive/MyDrive/nlp_news_pipeline/data/raw/gdelt_events_2025.csv
1500 URLs left to fetch for 2025.


ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.traveldailynews.com/aviation/balpa-secures-landmark-court-victory-against-ryanair-over-illegal-blacklisting-practices/
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.wgrz.com/article/news/local/wife-of-western-new-york-man-released-by-taliban-excited-for-his-return/71-13b8acdd-b752-478b-b771-d030449d984a


2025: fetching:   0%|          | 0/1500 [00:00<?, ?it/s]

ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.oneindia.com/international/donald-melania-trump-command-attention-at-commander-in-chief-inaugural-ball-with-elegant-dance-4048423.html
ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://grenadachronicle.com/why-is-trump-releasing-the-last-files-on-jfk-rfl-mlk-assassinations/
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.oregonlive.com/portland/2026/01/ice-detains-family-seeking-emergency-care-for-child-at-portland-hospital.html
ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://www.thewesterlysun.com/news/national/what-jfk-assassination-files-are-still-classified-trump-s-order-could-bring-them-to-light/article_1ca85226-c78e-5261-9fc1-bebc57bb6945.html
ERROR:trafilatura.downloads:download error: https://legalnewsline.com/stories/669117963-chilean-nationals-indicted-for-burglary-spree-in-southwest-ohio HTTPSConnectionPool(host='www.legalnewsline.com', port=4

2025: kept 1077/1500 rows with usable extracted text.
2025: 1052 unique-article clusters from 1077 raw articles.
2025: saved 1052 deduplicated rows to /content/drive/MyDrive/nlp_news_pipeline/data/processed/articles_deduped_2025.csv
sample_type
regular             761
targeted            213
regular+targeted     78
Name: count, dtype: int64
===== 2025 done =====



## Run 2026

In [10]:
result_2026 = process_year(2026)

===== Processing 2026 =====
10 sample weeks for 2026.
Running query for 2026 ...
Retrieved 458899 raw rows for 2026 (before per-week cap).


2026: tagging:   0%|          | 0/458899 [00:00<?, ?it/s]

Capping to at most 100 articles per week...


2026: capping:   0%|          | 0/10 [00:00<?, ?it/s]

2026: 1000 rows after capping (was 458899).
Saved to /content/drive/MyDrive/nlp_news_pipeline/data/raw/gdelt_events_2026.csv
1000 URLs left to fetch for 2026.


2026: fetching:   0%|          | 0/1000 [00:00<?, ?it/s]

ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://www.aol.com/articles/greenlands-furious-response-trumps-annexation-124916271.html
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://itbrief.co.nz/story/dxc-unveils-amber-to-speed-software-defined-vehicles
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.jugantor.com/international/1051499
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://finance.yahoo.com/news/japan-protests-china-export-controls-235611393.html
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML


2026: kept 798/1000 rows with usable extracted text.
2026: 791 unique-article clusters from 798 raw articles.
2026: saved 791 deduplicated rows to /content/drive/MyDrive/nlp_news_pipeline/data/processed/articles_deduped_2026.csv
sample_type
regular             555
targeted            159
regular+targeted     77
Name: count, dtype: int64
===== 2026 done =====



# After all years are done: combine into one final dataset

In [11]:
import glob

year_files = sorted(glob.glob(f'{PROJECT_DIR}/data/processed/articles_deduped_*.csv'))
print('Found year files:', year_files)

all_years_df = pd.concat([pd.read_csv(f) for f in year_files], ignore_index=True)
all_years_df = all_years_df.sort_values('date').reset_index(drop=True)

final_path = f'{PROJECT_DIR}/data/processed/articles_deduped_all_years.csv'
all_years_df.to_csv(final_path, index=False)
print(f'Combined {len(all_years_df)} rows across all years -> {final_path}')
print('\nBy sample_type:')
print(all_years_df['sample_type'].value_counts())
print('\nBy year:')
print(pd.to_datetime(all_years_df['date']).dt.year.value_counts().sort_index())

Found year files: ['/content/drive/MyDrive/nlp_news_pipeline/data/processed/articles_deduped_2021.csv', '/content/drive/MyDrive/nlp_news_pipeline/data/processed/articles_deduped_2022.csv', '/content/drive/MyDrive/nlp_news_pipeline/data/processed/articles_deduped_2023.csv', '/content/drive/MyDrive/nlp_news_pipeline/data/processed/articles_deduped_2024.csv', '/content/drive/MyDrive/nlp_news_pipeline/data/processed/articles_deduped_2025.csv', '/content/drive/MyDrive/nlp_news_pipeline/data/processed/articles_deduped_2026.csv', '/content/drive/MyDrive/nlp_news_pipeline/data/processed/articles_deduped_all_years.csv']
Combined 7163 rows across all years -> /content/drive/MyDrive/nlp_news_pipeline/data/processed/articles_deduped_all_years.csv

By sample_type:
sample_type
regular             6335
targeted             673
regular+targeted     155
Name: count, dtype: int64

By year:
date
2021    2746
2022     873
2023     775
2024     877
2025    1057
2026     835
Name: count, dtype: int64
